In [1]:
!pip install -q gradio scikit-learn pandas numpy matplotlib

In [2]:

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, mean_squared_error, r2_score
)

def run_algorithm(algorithm, test_size, k, max_depth, c_value):
    test_size = float(test_size)

    if algorithm in ["Simple Linear Regression",
                     "Multiple Linear Regression"]:
        data = load_diabetes(as_frame=True)
        df = data.frame

        if algorithm == "Simple Linear Regression":
            X = df[[data.feature_names[0]]]
        else:
            X = df[data.feature_names]

        y = df["target"]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        model = LinearRegression()
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = (
            f"R² Score: {r2_score(y_test, pred):.3f}\n"
            f"MSE: {mean_squared_error(y_test, pred):.2f}"
        )
        results = pd.DataFrame({
            "Actual": y_test.to_numpy(),
            "Predicted": pred
        }).head(20)

        fig, ax = plt.subplots()
        ax.scatter(y_test, pred)
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        ax.set_title("Actual vs Predicted")
        return metrics, results, fig

    data = load_iris(as_frame=True)
    X, y = data.data, data.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    if algorithm == "Logistic Regression":
        model = make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=1000)
        )
    elif algorithm == "KNN Classifier":
        model = make_pipeline(
            StandardScaler(), KNeighborsClassifier(n_neighbors=int(k))
        )
    elif algorithm == "Naive Bayes":
        model = GaussianNB()
    elif algorithm == "Decision Tree":
        model = DecisionTreeClassifier(max_depth=int(max_depth),
                                       random_state=42)
    else:
        model = make_pipeline(StandardScaler(), SVC(C=float(c_value)))

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    metrics = (
        f"Accuracy: {accuracy_score(y_test, pred) * 100:.2f}%\n\n"
        + classification_report(
            y_test, pred, target_names=data.target_names,
            zero_division=0
        )
    )

    results = pd.DataFrame({
        "Actual": data.target_names[y_test.to_numpy()],
        "Predicted": data.target_names[pred]
    })

    cm = confusion_matrix(y_test, pred)
    fig, ax = plt.subplots()
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(data.target_names, rotation=25)
    ax.set_yticklabels(data.target_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")

    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    return metrics, results, fig


with gr.Blocks(title="Integrated ML Dashboard") as demo:
    gr.Markdown("# Integrated Machine Learning Algorithm Dashboard")
    gr.Markdown("Select an algorithm, set parameters, and run it.")

    with gr.Row():
        algorithm = gr.Dropdown(
            choices=[
                "Simple Linear Regression",
                "Multiple Linear Regression",
                "Logistic Regression",
                "KNN Classifier",
                "Naive Bayes",
                "Decision Tree",
                "SVM Classifier"
            ],
            value="Naive Bayes",
            label="Select Algorithm"
        )

        test_size = gr.Slider(
            minimum=0.1, maximum=0.4, value=0.2,
            step=0.05, label="Test Data Proportion"
        )

    with gr.Row():
        k = gr.Slider(1, 15, value=5, step=1,
                      label="K for KNN")
        max_depth = gr.Slider(1, 20, value=5, step=1,
                              label="Decision Tree Depth")
        c_value = gr.Slider(0.1, 10, value=1, step=0.1,
                            label="SVM C Value")

    run_button = gr.Button("Run Algorithm", variant="primary")
    metrics = gr.Textbox(label="Performance Metrics", lines=12)
    results = gr.Dataframe(label="Prediction Results")
    plot = gr.Plot(label="Visualization")

    run_button.click(
        fn=run_algorithm,
        inputs=[algorithm, test_size, k, max_depth, c_value],
        outputs=[metrics, results, plot]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1282583fa7427df49e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
